# IMC2024 — LoMa w/ ALIKED

In [ ]:
from pathlib import Path
import os, shutil, sys

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
os.environ.setdefault('XDG_CACHE_HOME', '/tmp/xdg-cache')

DEPS_ROOT = Path('/kaggle/input/datasets/holdtensec/deps-prev-best')
DEPS_DIR  = DEPS_ROOT / 'deps'
LOMA_ROOT = DEPS_DIR / 'LoMa'

os.environ['_WHEEL_DIR'] = str(DEPS_DIR)
!pip install --no-index --no-deps --find-links $_WHEEL_DIR \
    pycolmap h5py kornia kornia-rs kneed lightglue \
    omegaconf antlr4-python3-runtime PyYAML typing-extensions einops

import torch

# Populate torch hub cache via symlinks (no disk duplication)
_hub = Path('/tmp/xdg-cache/torch/hub/checkpoints')
_hub.mkdir(parents=True, exist_ok=True)

WEIGHTS = {
    'loma_B.pt':                ['loma_B.pt'],
    'dad.pth':                  ['dad.pth'],
    'dedode_descriptor_G.pth':  ['dedode_descriptor_G.pth'],
    'aliked-n16.pth':           ['aliked-n16.pth'],
    'aliked_lightglue.pth':     ['aliked_lightglue_v0-1_arxiv-pth'],
    'dinov2_vitl14_pretrain.pth': ['dinov2_vitl14_pretrain.pth'],
}
for dep, aliases in WEIGHTS.items():
    origin = DEPS_DIR / dep
    assert origin.is_file(), f'Weight not found: {origin}'
    for alias in aliases:
        target = _hub / alias
        if target.exists() or target.is_symlink():
            target.unlink()
        os.symlink(origin, target)

print(f'{sum(len(a) for a in WEIGHTS.values())} weights linked into {_hub}')

In [ ]:
import gc, cv2, torch
import numpy as np, pandas as pd
from glob import glob
from tqdm import tqdm
import multiprocessing
import kornia.feature as KF
from torchmetrics import StructuralSimilarityIndexMeasure
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
from lightglue import ALIKED
from lightglue.utils import load_image

INPUT_ROOT  = '/kaggle/input/image-matching-challenge-2024'
OUTPUT_ROOT = '/kaggle/working'
DEBUG = False

In [ ]:
if DEBUG:
    scenes = ["transp_obj_glass_cylinder"]
    categories_df = pd.read_csv(f"{INPUT_ROOT}/train/categories.csv")
else:
    scenes = [x for x in os.listdir(f"{INPUT_ROOT}/test/") if os.path.isdir(f"{INPUT_ROOT}/test/{x}")]
    categories_df = pd.read_csv(f"{INPUT_ROOT}/test/categories.csv")

In [ ]:
image_sizes = [1024, 1280, 1600]

aliked_extractor = ALIKED(max_num_keypoints=4096, detection_threshold=0.01, resize=None).cuda().eval()

lightglue_matcher_params = {
    "filter_threshold": 0.2,
    "width_confidence": -1,
    "depth_confidence": -1,
    "mp": True
}
lightglue_matcher = KF.LightGlueMatcher(feature_name="aliked", params=lightglue_matcher_params).cuda().eval()

ssim_cuda = StructuralSimilarityIndexMeasure(data_range=255.).cuda()

In [ ]:
def get_rmats(n):
    ux, uy, uz = 0, 0, 1

    thetas = [(360 / n) * i for i in range(n)]
    Rmats = []
    for theta in thetas:
        costheta = np.cos(np.deg2rad(theta)).round(8)
        sintheta = np.sin(np.deg2rad(theta)).round(8)

        r00 = costheta + ux**2*(1-costheta)
        r01 = ux * uy * (1-costheta) - uz*sintheta
        r02 = ux * uz * (1-costheta) + uy*sintheta

        r10 = uy * ux * (1-costheta) + uz*sintheta
        r11 = costheta + uy**2*(1-costheta)
        r12 = uy * uz * (1-costheta) - ux*sintheta

        r20 = uz * ux * (1-costheta) - uy*sintheta
        r21 = uz * uy * (1-costheta) + ux*sintheta
        r22 = costheta + uz**2*(1-costheta)

        Rmat = np.array([[r00, r01, r02], [r10, r11, r12], [r20, r21, r22]])
        Rmats.append(Rmat)

    return Rmats


def get_distance_matrix(fnames, flows):
    distance_matrix = np.zeros((len(fnames), len(fnames)), dtype=np.int32)
    for idx1, fname1 in enumerate(fnames):
        for idx2, fname2 in enumerate(fnames):
            if idx1 == idx2:
                value = 0
            else:
                value = flows[(fname1, fname2)]
            distance_matrix[idx1, idx2] = value
    return distance_matrix


def get_distance_matrix_2(fnames, order_idxs_list):

    neighbors_dict = {idx: [] for idx in range(len(fnames))}
    for order_idxs in order_idxs_list:
        for idx in range(len(fnames)):
            neighbor_left = order_idxs[idx-1]
            neighbor_right = order_idxs[0] if idx == len(fnames) - 1 else order_idxs[idx+1]
            neighbors_dict[order_idxs[idx]].extend([neighbor_left, neighbor_right])

    flows = {}
    for i in range(len(fnames)):
        for j in range(len(fnames)):
            if i == j:
                value = 0
            else:
                cnt = neighbors_dict[i].count(j)
                if cnt == 0:
                    value = 100000000
                else:
                    value = 1000000 // cnt
            flows[(i, j)] = value

    distance_matrix = np.zeros((len(fnames), len(fnames)), dtype=np.int32)
    for idx1 in range(len(fnames)):
        for idx2 in range(len(fnames)):
            distance_matrix[idx1, idx2] = flows[(idx1, idx2)]

    dummy_distance_matrix = np.zeros((len(fnames)+1, len(fnames)+1), dtype=np.int32)
    dummy_distance_matrix[1:,1:] = distance_matrix
    return dummy_distance_matrix


def tsp_distance_callback(from_index, to_index):
    """Returns the distance between the two nodes."""
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return distance_matrix[from_node][to_node]


def get_tsp_solution(manager, routing, solution):
    """Prints solution on console."""
    index = routing.Start(0)
    idxs = []
    while not routing.IsEnd(index):
        idxs.append(manager.IndexToNode(index))
        index = solution.Value(routing.NextVar(index))
    return np.array(idxs)


def get_img_pairs_all(fnames):
    index_pairs = []
    for i in range(len(fnames)):
        for j in range(i+1, len(fnames)):
            index_pairs.append((i,j))
    return index_pairs


def resize(image, image_size):
    h, w = image.shape[:2]
    aspect_ratio = h/w
    smaller_side_size = int(image_size/max(aspect_ratio, 1/aspect_ratio))
    if aspect_ratio > 1: # H > W
        new_size = (image_size, smaller_side_size)
    else: # H <= W
        new_size = (smaller_side_size, image_size)
    image = cv2.resize(image, new_size[::-1])
    return image, new_size


def alg_inference(cache, fname1, fname2, image_size, rot_code_2):

    # Extraction
    if 'keypoints_alg' not in cache[fname1][image_size][0]:
        with torch.inference_mode():
            tensor = cache[fname1][image_size]['tensor_alg'].cuda()
            pred = aliked_extractor.extract(tensor, resize=None)
            cache[fname1][image_size][0] = {
                **cache[fname1][image_size][0],
                **{
                    'keypoints_alg': pred['keypoints'],
                    'descriptors_alg': pred['descriptors']
                }
            }

    if 'keypoints_alg' not in cache[fname2][image_size][rot_code_2]:
        with torch.inference_mode():
            tensor = torch.rot90(cache[fname2][image_size]['tensor_alg'], rot_code_2, [1, 2]).cuda()
            pred = aliked_extractor.extract(tensor, resize=None)
            cache[fname2][image_size][rot_code_2] = {
                **cache[fname2][image_size][rot_code_2],
                **{
                    'keypoints_alg': pred['keypoints'],
                    'descriptors_alg': pred['descriptors']
                }
            }

    # Matching
    kpts1, kpts2 = cache[fname1][image_size][0]['keypoints_alg'], cache[fname2][image_size][rot_code_2]['keypoints_alg']
    with torch.inference_mode():
        _, indices = lightglue_matcher(
            cache[fname1][image_size][0]['descriptors_alg'][0], 
            cache[fname2][image_size][rot_code_2]['descriptors_alg'][0],
            KF.laf_from_center_scale_ori(kpts1),
            KF.laf_from_center_scale_ori(kpts2)
        )
        kpts1 = kpts1[0].cpu().numpy()
        kpts2 = kpts2[0].cpu().numpy()
        indices = indices.cpu().numpy()
        
    mkpts1 = kpts1[indices[..., 0]].astype(np.float32)
    mkpts2 = kpts2[indices[..., 1]].astype(np.float32)

    try:
        _, inliers = cv2.findFundamentalMat(mkpts1, mkpts2, cv2.USAC_MAGSAC, ransacReprojThreshold=5, confidence=0.9999, maxIters=50000)
        inliers = inliers.ravel() > 0
        mkpts1 = mkpts1[inliers]
        mkpts2 = mkpts2[inliers]
    except:
        pass

    num_matches = len(mkpts1)

    return num_matches


def matching_inference(fname1, fname2, cache=None):

    for fname in [fname1, fname2]:
        if fname not in cache:
            img = read_image(fname)
            h, w = img.shape[:2]
            cache[fname] = {'image': img, 'h': h, 'w': w} 
            for image_size in image_sizes:
                if max(h, w) != image_size:
                    img_r, (h_r, w_r) = resize(img, image_size)
                else:
                    img_r = img.copy()
                    h_r, w_r = img_r.shape[:2]
                tensor_alg = numpy_image_to_torch(img_r)
                cache[fname][image_size] = {'tensor_alg': tensor_alg, 'h_r': h_r, 'w_r': w_r, 0: {}, 1: {}, 2: {}, 3: {}}

    # Co-orientation
    rot_code_2, max_n = 0, 0
    for rc2 in range(4):
        n = alg_inference(cache, fname1, fname2, image_sizes[0], rot_code_2=rc2)
        if n > max_n:
            rot_code_2 = rc2
            max_n = n

    num_matches = max_n
    for image_size in image_sizes[1:]:
        num_matches += alg_inference(cache, fname1, fname2, image_size, rot_code_2)

    return num_matches


def get_matching_flows(fnames):
    index_pairs = get_img_pairs_all(fnames=fnames)
    cache, flows = {}, {}
    for pair_idx in tqdm(index_pairs, desc="Matching"):
        idx1, idx2 = pair_idx
        fname1, fname2 = fnames[idx1], fnames[idx2]
        num_matches = matching_inference(fname1, fname2, cache)
        flows[(fname1, fname2)] = flows[(fname2, fname1)] = int((1 / num_matches) * 1e8)
    return flows


def compute_ssim(im1, im2):
    with torch.inference_mode():
        tensor1 = torch.tensor(im1.copy(), dtype=torch.float32)[None][None].cuda()
        tensor2 = torch.tensor(im2.copy(), dtype=torch.float32)[None][None].cuda()
        score = ssim_cuda(tensor1, tensor2).cpu().numpy().item()
    return score


def get_flows(fnames):
    relative_orientations = {fnames[0]: 0}
    flow_images = [cv2.imread(fname, cv2.IMREAD_GRAYSCALE) for fname in fnames]
    flow_images = [resize(image, 2048)[0] for image in flow_images]
    ssim_flows = {}
    for i in tqdm(range(len(fnames)), desc="Getting flows"):
        image_i = flow_images[i]
        for j in range(i+1, len(fnames)):
            image_j = flow_images[j]
            if i == 0:
                diffs = []
                try:
                    diffs.append(compute_ssim(image_i, image_j))
                except:
                    diffs.append(0)
                for rot_code in range(1,4):
                    try:
                        diffs.append(compute_ssim(image_i, np.rot90(image_j, rot_code)))
                    except:
                        diffs.append(0)
                min_diff_idx = np.argmax(diffs)
                relative_orientations[fnames[j]] = min_diff_idx
            
            r1, r2 = relative_orientations[fnames[i]], relative_orientations[fnames[j]]
            im1 = np.rot90(image_i, r1) if r1 else image_i
            im2 = np.rot90(image_j, r2) if r2 else image_j

            ssim_score = 1 - compute_ssim(im1, im2)
            ssim_flows[(fnames[i], fnames[j])] = ssim_flows[(fnames[j], fnames[i])] = ssim_score

    return ssim_flows


def arr_to_str(a):
    return ';'.join([str(x) for x in a.reshape(-1)])

In [ ]:
results_df = pd.DataFrame(columns=['image_path', 'dataset', 'scene', 'rotation_matrix', 'translation_vector'])
non_transparent_scenes = []
for scene in tqdm(scenes, desc='Running pipeline'):

    torch.cuda.empty_cache()
    gc.collect()
    
    categories = categories_df[categories_df["scene"]==scene]["categories"].item()
    use_crops = ("symmetr" not in categories) and ("transparen" not in categories)
    is_transparent = ("transparen" in categories)

    print(f"{scene=} {categories=} {use_crops=} {is_transparent=}")

    img_dir = f"{INPUT_ROOT}/{'train' if DEBUG else 'test'}/{scene}/images"
    if not os.path.exists(img_dir):
        continue

    fnames = sorted(glob(f"{img_dir}/*"))
    
    if is_transparent:
        Rmats = get_rmats(len(fnames))
    
        ssim_flows = get_flows(fnames)
        matching_flows = get_matching_flows(fnames)

        ssim_min, ssim_max = min(list(ssim_flows.values())), max(list(ssim_flows.values()))
        matching_min, matching_max = min(list(matching_flows.values())), max(list(matching_flows.values()))

        ssim_flows = {k: (v - ssim_min) / (ssim_max - ssim_min) for k, v in ssim_flows.items()}
        matching_flows = {k: (v - matching_min) / (matching_max - matching_min) for k, v in matching_flows.items()}

        flows = {k:int((ssim_flows[k] + matching_flows[k]) * 1e6) for k in ssim_flows.keys()}
        
        distance_matrix = get_distance_matrix(fnames, flows)
        
        ############# N + 1 #############
        order_idxs_list = []
        for start_idx in range(len(fnames)):
            manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, start_idx)
            routing = pywrapcp.RoutingModel(manager)
            transit_callback_index = routing.RegisterTransitCallback(tsp_distance_callback)
            routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
            search_parameters = pywrapcp.DefaultRoutingSearchParameters()
            search_parameters.first_solution_strategy = (routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
            solution = routing.SolveWithParameters(search_parameters)
            order_idxs = get_tsp_solution(manager, routing, solution)
            order_idxs_list.append(order_idxs)

        distance_matrix = get_distance_matrix_2(fnames, order_idxs_list)

        manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, 0)
        routing = pywrapcp.RoutingModel(manager)
        transit_callback_index = routing.RegisterTransitCallback(tsp_distance_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
        search_parameters = pywrapcp.DefaultRoutingSearchParameters()
        search_parameters.first_solution_strategy = (routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
        solution = routing.SolveWithParameters(search_parameters)
        order_idxs = get_tsp_solution(manager, routing, solution)
        order_idxs = order_idxs[1:] - 1
        ############# N + 1 #############
    else:
        non_transparent_scenes.append(scene)
        continue

    # Create submission
    for fid, fname in enumerate(fnames):
        image_id = '/'.join(fname.split('/')[-4:])
        if is_transparent:
            R = Rmats[np.where(order_idxs==fid)[0].item()]
            T = np.array([100, 100, 100])
        elif image_id in results:
            R = results[image_id]['R']
            T = results[image_id]['t']
        else:
            R = np.eye(3)
            T = np.zeros(3)

        new_row = pd.DataFrame({'image_path': image_id,
                                'dataset': scene,
                                'scene': scene,
                                'rotation_matrix': arr_to_str(R),
                                'translation_vector': arr_to_str(T)}, index=[0])

        results_df = pd.concat([results_df, new_row]).reset_index(drop=True)

In [ ]:
results_df.to_csv(f"{OUTPUT_ROOT}/transp_submission.csv", index=False)

In [ ]:
import json
with open("/kaggle/working/non_transparent_scenes.json", "w") as f:
    json.dump(non_transparent_scenes, f)

In [ ]:
aliked_extractor.cpu()
lightglue_matcher.cpu()
ssim_cuda.cpu()
del aliked_extractor, lightglue_matcher, ssim_cuda
torch.cuda.empty_cache()
gc.collect()

---
# Non-Transparent Part
## Prepare Inference Scripts

In [ ]:
!cp -r /kaggle/input/datasets/holdtensec/final-inf-scripts/new-inference-scripts /kaggle/working/imc2024-inference-scripts
!cp -r /kaggle/input/datasets/holdtensec/final-pipeline/new-pipelines /kaggle/working/new-pipelines

In [ ]:
%%writefile imc2024-inference-scripts/config.py

import os
import json

class Config:
    input_dir_root = "/kaggle/input"
    output_dir = "/tmp"
    check_exist_dir = False

    input_csv = "/kaggle/input/image-matching-challenge-2024/sample_submission.csv"
    category_csv = "/kaggle/input/image-matching-challenge-2024/test/categories.csv"
    target_datasets = []

    pipeline_json = "/kaggle/working/new-pipelines/exp_ver58_loma_ensemble/pipeline.json"
    transparent_pipeline_json = "/kaggle/working/new-pipelines/exp_ver58_loma_ensemble/transp_pipeline.json"

    colmap_mapper_options = {
        "min_model_size": 3,
        "max_num_models": 3,
    }

In [ ]:
from pathlib import Path
import json as _json

PIPELINE_DIR = Path('/kaggle/working/new-pipelines/exp_ver58_loma_ensemble')
loma_code_root = str(LOMA_ROOT / 'src')
loma_weights = str(Path('/tmp/xdg-cache/torch/hub/checkpoints/loma_B.pt'))

for jf in PIPELINE_DIR.glob('*.json'):
    text = jf.read_text()
    text = text.replace('__LOMA_CODE_ROOT__', loma_code_root)
    text = text.replace('__LOMA_WEIGHTS_PATH__', loma_weights)
    jf.write_text(text)
    print(f'patched {jf.name}')

## Split scene list

In [ ]:
root_path = f"{INPUT_ROOT}/test"
data_num_list = [sum(len(files) for _, _, files in os.walk(f"{root_path}/{scene}/images")) for scene in non_transparent_scenes]
data_num_list

In [ ]:
if 0 in data_num_list:
    raise Exception("Found scene with 0 images")

In [ ]:
def partition_with_index(lst, lst2):
    indexed_lst = sorted(enumerate(lst), key=lambda x: x[1], reverse=True)
    groupA, groupB, groupA2, groupB2 = [], [], [], []
    for index, item in indexed_lst:
        if sum([lst[i] for i in groupA]) <= sum([lst[i] for i in groupB]):
            groupA.append(index); groupA2.append(lst2[index])
        else:
            groupB.append(index); groupB2.append(lst2[index])
    return groupA2, groupB2

non_transparent_scenes0, non_transparent_scenes1 = partition_with_index(data_num_list, non_transparent_scenes)
print(non_transparent_scenes0)
print(non_transparent_scenes1)

## Exec Inference

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
def make_command(device_id, target_datasets):
    cmd = f"python imc2024-inference-scripts/inference.py --device_id {device_id} --output_dir /tmp/tmp{device_id}"
    if len(target_datasets) > 0:
        cmd += " --target_datasets"
        for scene in target_datasets:
            cmd += f" {scene}"
    return cmd.split(" ")

In [ ]:
import subprocess
proc0 = subprocess.Popen(make_command(0, non_transparent_scenes0))
proc1 = subprocess.Popen(make_command(1, non_transparent_scenes1))
proc0.wait()
proc1.wait()

In [ ]:
df1 = pd.read_csv("/tmp/tmp0/submission.csv")
df2 = pd.read_csv("/tmp/tmp1/submission.csv")
df = pd.concat([df1, df2]).reset_index(drop=True)
df.to_csv("/tmp/submission.csv", index=False)

## Merge csv

In [ ]:
df1 = pd.read_csv(f"{OUTPUT_ROOT}/transp_submission.csv")
df2 = pd.read_csv("/tmp/submission.csv")
df1 = df1.query("scene not in @non_transparent_scenes").reset_index(drop=True)
df = pd.concat([df1, df2])
FINAL_SUB_TMP_PATH = "/tmp/final_submission.csv"
df.to_csv(FINAL_SUB_TMP_PATH, index=False)
print(df)

In [ ]:
%%writefile imc2024-inference-scripts/config.py

import os
import json

class Config:
    input_dir_root = "/kaggle/input"
    output_dir = "/tmp"
    check_exist_dir = False

    input_csv = "/kaggle/input/image-matching-challenge-2024/sample_submission.csv"
    category_csv = "/kaggle/input/image-matching-challenge-2024/test/categories.csv"
    target_datasets = []

    pipeline_json = "/kaggle/working/new-pipelines/exp_ver58_loma_ensemble/pipeline_fallback.json"
    transparent_pipeline_json = "/kaggle/working/new-pipelines/exp_ver58_loma_ensemble/transp_pipeline.json"

    colmap_mapper_options = {
        "min_model_size": 3,
        "max_num_models": 2,
    }

In [ ]:
thr_null_img = 0.15

In [ ]:
df1_ = pd.read_csv(FINAL_SUB_TMP_PATH)
df_non_trsp = df1_.query("scene in @non_transparent_scenes").reset_index(drop=True)
print(df_non_trsp.shape)

rot_m_null = '1.0;0.0;0.0;0.0;1.0;0.0;0.0;0.0;1.0'
tr_v_null = '0.0;0.0;0.0'

scene_cnt_dict = df_non_trsp.groupby(['scene']).image_path.count().to_dict()
print('scene_cnt_dict', scene_cnt_dict)

mask_null = (df_non_trsp['rotation_matrix'] == rot_m_null) & (df_non_trsp['translation_vector'] == tr_v_null)
scene_cnt_null_dict = df_non_trsp[mask_null].groupby(['scene']).image_path.count().to_dict()

poor_non_trsp_scenes = [k for k, v in scene_cnt_null_dict.items() if v / scene_cnt_dict[k] >= thr_null_img]
print('poor_non_trsp_scenes', poor_non_trsp_scenes)

In [ ]:
data_num_list = [sum(len(files) for _, _, files in os.walk(f"{root_path}/{scene}/images")) for scene in poor_non_trsp_scenes]
data_num_list

In [ ]:
non_transparent_scenes0, non_transparent_scenes1 = partition_with_index(data_num_list, poor_non_trsp_scenes)
print(non_transparent_scenes0)
print(non_transparent_scenes1)

In [ ]:
!rm -rf /tmp/tmp0 /tmp/tmp1

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
import subprocess
proc0 = subprocess.Popen(make_command(0, non_transparent_scenes0))
proc1 = subprocess.Popen(make_command(1, non_transparent_scenes1))
proc0.wait()
proc1.wait()

In [ ]:
df1_ = pd.read_csv("/tmp/tmp0/submission.csv")
df2_ = pd.read_csv("/tmp/tmp1/submission.csv")
df_poor = pd.concat([df1_, df2_]).reset_index(drop=True)
df_poor.to_csv("/tmp/submission_poor.csv", index=False)

In [ ]:
df_poor = pd.read_csv("/tmp/submission_poor.csv")

mask_null = (df_non_trsp['rotation_matrix']==rot_m_null) & (df_non_trsp['translation_vector']==tr_v_null)
scene_cnt_reg_dict = df_non_trsp[~mask_null].groupby(['scene']).image_path.count().to_dict()

mask_poor_null = (df_poor['rotation_matrix']==rot_m_null) & (df_poor['translation_vector']==tr_v_null)
scene_poor_cnt_reg_dict = df_poor[~mask_poor_null].groupby(['scene']).image_path.count().to_dict()

replace_rec_scenes = [k for k, v in scene_poor_cnt_reg_dict.items() if v > scene_cnt_reg_dict.get(k, 0)]
print('Update reconstructions with', replace_rec_scenes)

df_non_trsp_new = df_non_trsp[~df_non_trsp.scene.isin(replace_rec_scenes)]
df_poor_new = df_poor[df_poor.scene.isin(replace_rec_scenes)]
df_non_trnsp_ = pd.concat([df_non_trsp_new, df_poor_new]).reset_index(drop=True)
df_non_trnsp_.to_csv("/tmp/submission.csv", index=False)

In [ ]:
if len(replace_rec_scenes) > 0:
    df1_ = pd.read_csv(f"{OUTPUT_ROOT}/transp_submission.csv")
    df1_ = df1_.query("scene not in @non_transparent_scenes").reset_index(drop=True)
    df2_ = pd.read_csv("/tmp/submission.csv")
    df = pd.concat([df1_, df2_])
    df.to_csv(FINAL_SUB_TMP_PATH, index=False)
    print(df)

In [ ]:
FINAL_SUB_PATH = f"{OUTPUT_ROOT}/submission.csv"
!cp {FINAL_SUB_TMP_PATH} {FINAL_SUB_PATH}